# 07. cal01_base_v2 만들기

| 항목 | 내용 |
| --- | --- |
| 입력 | `cal01_base_v1.csv`, 검토 표 `data/labels/domain_decisions.csv` |
| 사용하는 검토 | 검토 표의 VERSION이 v2인 행 |
| 출력 | `data/processed/cal01_base_v2.csv` (원기록 345,390행) |
| 이 버전에서 하는 일 | v1에 공통 움직임 검토로 해제한 후보를 `DOMAIN_LABEL=0`으로 붙임 |

**라벨 규칙**

| 컬럼 | 1 | 0 | 빈칸 |
| --- | --- | --- | --- |
| `구역__STAT_LABEL` | v1과 같음 | v1과 같음 | v1과 같음 |
| `구역__DOMAIN_LABEL` | 검토 표에서 이상으로 확정한 구간 | \|Z\| > 3인 구역 센서가 모두 검토로 해제됨 | 검토하지 않음 |

- 검토 표 한 줄 = LOT, 센서, 시작, 끝(포함), 0 또는 1, 이유
- 센서 하나만 해제되고 같은 구역의 다른 센서가 아직 \|Z\| > 3이면 0으로 바꾸지 않음

**v1 ~ v5 한눈에 보기** (버전마다 한 가지씩만 더함)

| 버전 | 단위 | STAT_LABEL | DOMAIN_LABEL | 만드는 노트북 |
| --- | --- | --- | --- | --- |
| v1 | 원기록 (1초) | 같은 LOT 기준 \|Z\| > 3 | 없음 | 06 |
| v2 | 원기록 (1초) | v1과 같음 | 공통 움직임 검토로 해제한 곳 0 | 07 |
| v3 | 원기록 (1초) | v1과 같음 | v2 + 새로 찾은 공통 움직임 해제 0 | 08 |
| v4 | 1분 | 1분 안에 후보가 하나라도 있으면 1 | v2를 1분으로 묶음 | 09 |
| v5 | 1분 | v4와 같음 | v3를 1분으로 묶음 | 10 |

## 1. 공통 경로

In [ ]:
import os
import pandas as pd

# 저장소 루트 찾기: notebooks 폴더가 안 보이면 한 칸 위로 (최대 두 번)
ROOT = os.getcwd()
for step in range(2):
    if not os.path.isdir(os.path.join(ROOT, "notebooks")):
        ROOT = os.path.dirname(ROOT)

# 폴더
DATA_DIR = os.path.join(ROOT, "data")                    # 원본 CSV 8개
if os.path.isfile(os.path.join(DATA_DIR, "raw", "T-CR1-CAL01_온도.csv")):
    DATA_DIR = os.path.join(DATA_DIR, "raw")             # data/raw에 두었으면 그 폴더를 씀
PROCESSED_DIR = os.path.join(ROOT, "data", "processed")  # 만든 CSV (cal01_base_v1 ~ v5)
LABEL_DIR = os.path.join(ROOT, "data", "labels")         # 팀 검토 표 (domain_decisions.csv)
FIG_DIR = os.path.join(ROOT, "figures")                  # 그림
ENC = "utf-8-sig"

# 원본 CSV
TEMP_CSV = os.path.join(DATA_DIR, "T-CR1-CAL01_온도.csv")
EVENT_CSV = os.path.join(DATA_DIR, "G-02_조업이벤트.csv")
REPAIR_CSV = os.path.join(DATA_DIR, "G-01_정기수리캘린더.csv")
EQP_CSV = os.path.join(DATA_DIR, "G-01_설비마스터.csv")
PART_CSV = os.path.join(DATA_DIR, "G-01_부품마스터.csv")
MAINT_CSV = os.path.join(DATA_DIR, "G-02_정비이력.csv")
PARTCHG_CSV = os.path.join(DATA_DIR, "G-02_부품교체이력.csv")
WELD_CSV = os.path.join(DATA_DIR, "T-CR1-WLD01_용접.csv")

# 읽을 가공 CSV (이름만 바꾸면 됨)
DATA_FILE = "cal01_base_v1.csv"  # v1을 읽음
BASE_CSV = os.path.join(PROCESSED_DIR, DATA_FILE)

# 만들 CSV
OUTPUT_FILE = "cal01_base_v2.csv"
OUT_CSV = os.path.join(PROCESSED_DIR, OUTPUT_FILE)

# 검토 표와, 그중 사용할 VERSION
DECISION_CSV = os.path.join(LABEL_DIR, "domain_decisions.csv")
USE_VERSIONS = ["v2"]

import numpy as np

## 2. 구역과 센서

In [ ]:
# 구역별 센서 (CP-Z4M은 값이 0.0000000001 수준이라 라벨 계산에 쓰지 않음)
GROUPS = {
    "CLN": ["CLN-IN"],
    "Z1": ["TC-Z1", "OP-Z1"],
    "Z2": ["TC-Z2", "OP-Z2"],
    "Z3": ["TC-Z3", "OP-Z3"],
    "Z4": ["TC-Z4", "OP-Z4", "CP-Z4"],
    "Z5": ["TC-Z5", "OP-Z5"],
    "Z6": ["TC-Z6", "OP-Z6"],
    "Z7": ["TC-Z7", "OP-Z7"],
    "Z8": ["TC-Z8", "OP-Z8"],
    "Z9": ["TC-Z9"],
    "COOLING": ["TC-OUT1", "TC-OUT2"],
}

# 라벨 계산에 쓰는 센서 21개
SENSORS = []
for zone in GROUPS:
    SENSORS = SENSORS + GROUPS[zone]

# CSV에 저장하는 원본 센서 22개 (CP-Z4M은 저장만 하고 계산에는 안 씀)
SAVE_SENSORS = SENSORS + ["CP-Z4M"]

# 라벨 컬럼 22개 (구역마다 STAT, DOMAIN)
LABEL_COLUMNS = []
for zone in GROUPS:
    LABEL_COLUMNS.append(zone + "__STAT_LABEL")
    LABEL_COLUMNS.append(zone + "__DOMAIN_LABEL")

## 3. v1과 검토 표 읽기

In [ ]:
base = pd.read_csv(BASE_CSV, encoding=ENC, low_memory=False)  # low_memory=False: 컬럼 형식을 파일 전체를 보고 정함
base["MEAS_DT"] = pd.to_datetime(base["MEAS_DT"], errors="raise")

decisions = pd.read_csv(DECISION_CSV, encoding=ENC)
decisions["START"] = pd.to_datetime(decisions["START"], errors="raise")
decisions["END"] = pd.to_datetime(decisions["END"], errors="raise")

# 이 버전에서 쓸 검토만 남김
decisions = decisions[decisions["VERSION"].isin(USE_VERSIONS)].reset_index(drop=True)

print("데이터:", base.shape)
decisions.groupby(["VERSION", "SOURCE", "DOMAIN_LABEL"]).size()

## 4. 센서별 Z-score (v1과 같은 계산)

In [ ]:
# 제외 행을 뺀 데이터로 LOT별 평균, 표준편차를 구함
use = base[base["EXCLUDED"] == False]
lot_mean = use.groupby("LOT_NO")[SENSORS].mean()
lot_std = use.groupby("LOT_NO")[SENSORS].std(ddof=0)

# 센서마다 |Z| = |값 - LOT 평균| / LOT 표준편차
z = pd.DataFrame(np.nan, index=base.index, columns=SENSORS)

for sensor in SENSORS:
    mean = base["LOT_NO"].map(lot_mean[sensor])
    std = base["LOT_NO"].map(lot_std[sensor])
    std = std.replace(0, np.nan)

    z[sensor] = (base[sensor] - mean).abs() / std

# 제외 행은 Z를 비워둠
z.loc[base["EXCLUDED"] == True] = np.nan

## 5. 검토 구간 표시하기

센서마다 두 가지 표를 만듦.

| 표 | True인 곳 |
| --- | --- |
| `cleared` | 검토해서 후보를 해제한 행 (DOMAIN 0) |
| `confirmed` | 검토해서 이상으로 확정한 행 (DOMAIN 1) |

In [ ]:
cleared = pd.DataFrame(False, index=base.index, columns=SENSORS)
confirmed = pd.DataFrame(False, index=base.index, columns=SENSORS)

for i in range(len(decisions)):
    row = decisions.loc[i]

    same_lot = base["LOT_NO"] == row["LOT_NO"]
    in_time = (base["MEAS_DT"] >= row["START"]) & (base["MEAS_DT"] <= row["END"])
    hit = same_lot & in_time

    if row["DOMAIN_LABEL"] == 0:
        cleared.loc[hit, row["SENSOR"]] = True
    else:
        confirmed.loc[hit, row["SENSOR"]] = True

print("해제 표시가 있는 행:", int(cleared.any(axis=1).sum()))
print("확정 표시가 있는 행:", int(confirmed.any(axis=1).sum()))

## 6. 구역별 DOMAIN_LABEL 붙이기

| 상황 | DOMAIN_LABEL |
| --- | --- |
| 구역 센서 중 하나라도 확정 구간에 있음 | 1 |
| STAT 1이고, \|Z\| > 3인 구역 센서가 모두 해제됨 | 0 |
| 그 밖 | 빈칸 |

In [ ]:
for zone in GROUPS:
    zone_sensors = GROUPS[zone]
    stat_col = zone + "__STAT_LABEL"
    domain_col = zone + "__DOMAIN_LABEL"

    # |Z| > 3인데 아직 해제되지 않은 센서가 남아 있는지
    over = z[zone_sensors] > 3
    still_open = (over & (cleared[zone_sensors] == False)).any(axis=1)

    is_candidate = base[stat_col] == 1
    is_known = base[stat_col].notna()
    is_confirmed = confirmed[zone_sensors].any(axis=1)

    base[domain_col] = np.nan
    base.loc[is_candidate & (still_open == False), domain_col] = 0
    base.loc[is_known & is_confirmed, domain_col] = 1

## 7. 저장 (컬럼은 v1과 같음)

In [ ]:
# 라벨을 1.0 대신 1로 저장하려고 정수형(Int64)으로 바꿈. Int64는 빈칸도 담을 수 있음
for column in LABEL_COLUMNS:
    base[column] = base[column].astype("Int64")

base.to_csv(OUT_CSV, index=False, encoding=ENC)
print("저장:", OUTPUT_FILE, base.shape)

## 8. 구역별 라벨 개수

In [ ]:
# 구역별 개수
rows = []

for zone in GROUPS:
    stat = base[zone + "__STAT_LABEL"]
    domain = base[zone + "__DOMAIN_LABEL"]

    rows.append({
        "구역": zone,
        "STAT 1": int((stat == 1).sum()),
        "STAT 0": int((stat == 0).sum()),
        "STAT 빈칸": int(stat.isna().sum()),
        "DOMAIN 1": int((domain == 1).sum()),
        "DOMAIN 0": int((domain == 0).sum()),
    })

zone_counts = pd.DataFrame(rows)
zone_counts

## 9. LOT별로 해제된 행 수

In [ ]:
domain_columns = []
for zone in GROUPS:
    domain_columns.append(zone + "__DOMAIN_LABEL")

cleared_by_lot = (base[domain_columns] == 0).groupby(base["LOT_NO"]).sum()
cleared_by_lot.columns = list(GROUPS)
cleared_by_lot